# 04 - Gold Preparatória para BI

## Objetivo

Criar uma camada semântica detalhada a partir da tabela Silver, preparada para consumo por ferramentas de Business Intelligence.

A tabela mantém o mesmo grão da Silver:

> Cada linha representa um registro de benefício concedido pelo INSS.

As transformações realizadas nesta etapa incluem:

- seleção e organização dos campos relevantes;
- criação de descrições amigáveis;
- criação de atributos temporais;
- cálculo da idade na competência;
- classificação por faixa etária;
- cálculo da duração do benefício;
- criação de indicadores auxiliares;
- classificação da natureza do afastamento;
- inclusão de uma medida unitária para contagem no BI.

A tabela gerada é:

`afastamento_inss.gold.prep_beneficios_bi`

## 1. Importações

Todas as funções utilizadas no notebook são importadas em uma única célula.

Esse padrão facilita:

- manutenção;
- revisão;
- conversão futura do notebook em script;
- identificação das dependências;
- execução sequencial do pipeline.

In [0]:
# ============================================================
# IMPORTACOES
# ============================================================

from pyspark.sql.functions import (
    col,
    when,
    lit,
    current_timestamp,
    concat,
    to_date,
    substring,
    split,
    trim,
    regexp_replace,
    months_between,
    floor,
    datediff,
    year,
    month,
    date_format,
    count,
    sum as spark_sum
)

## 2. Parâmetros

A tabela de origem corresponde à Silver validada.

A tabela de destino será criada no schema Gold e servirá como fonte detalhada para o modelo analítico e para futuras visualizações.

In [0]:
# ============================================================
# PARAMETROS
# ============================================================

TABELA_ORIGEM = (
    "afastamento_inss.silver.beneficios_concedidos"
)

TABELA_DESTINO = (
    "afastamento_inss.gold.prep_beneficios_bi"
)

print(f"Origem : {TABELA_ORIGEM}")
print(f"Destino: {TABELA_DESTINO}")

## 3. Leitura da camada Silver

A Gold deve ler exclusivamente dados tratados e tipados da Silver.

A leitura direta da Bronze ou do arquivo CSV quebraria a rastreabilidade e repetiria regras já aplicadas nas etapas anteriores.

In [0]:
df_silver = spark.table(TABELA_ORIGEM)

quantidade_linhas_silver = df_silver.count()
quantidade_colunas_silver = len(df_silver.columns)

print(f"Linhas : {quantidade_linhas_silver:,}")
print(f"Colunas: {quantidade_colunas_silver}")

## 4. Validação do contrato de entrada

Antes das transformações, o notebook verifica se todas as colunas necessárias estão disponíveis na tabela Silver.

Caso uma coluna seja renomeada ou removida em uma etapa anterior, a execução será interrompida com uma mensagem objetiva.

In [0]:
df_gold = (
    df_silver
    .withColumn(
        "dt_competencia",
        to_date(
            concat(
                col("competencia_concessao"),
                lit("01")
            ),
            "yyyyMMdd"
        )
    )
    .withColumn(
        "ano_competencia",
        year(col("dt_competencia"))
    )
    .withColumn(
        "mes_competencia",
        month(col("dt_competencia"))
    )
    .withColumn(
        "ano_mes_competencia",
        date_format(
            col("dt_competencia"),
            "yyyy-MM"
        )
    )
)

display(
    df_gold
    .select(
        "competencia_concessao",
        "dt_competencia",
        "ano_competencia",
        "mes_competencia",
        "ano_mes_competencia"
    )
    .distinct()
)

## 6. Idade na competência

A idade é calculada entre a data de nascimento e a data de referência da competência.

Como o arquivo apresenta apenas a competência mensal, e não uma data individual de concessão, o cálculo utiliza o primeiro dia do mês como referência.

A coluna deve ser interpretada como idade aproximada na competência.

In [0]:
df_gold = df_gold.withColumn(
    "idade_na_competencia",
    floor(
        months_between(
            col("dt_competencia"),
            col("dt_nascimento")
        ) / lit(12)
    )
)

display(
    df_gold
    .select(
        "dt_nascimento",
        "dt_competencia",
        "idade_na_competencia"
    )
    .limit(10)
)

In [0]:
idades_invalidas = df_gold.filter(
    (col("idade_na_competencia") < 0)
    | (col("idade_na_competencia") > 120)
).count()

idades_nulas = df_gold.filter(
    col("idade_na_competencia").isNull()
).count()

print("=" * 55)
print("VALIDACAO DA IDADE NA COMPETENCIA")
print("=" * 55)
print(f"Idades nulas             : {idades_nulas:,}")
print(f"Idades menores que zero  : "
      f"{df_gold.filter(col('idade_na_competencia') < 0).count():,}")
print(f"Idades maiores que 120   : "
      f"{df_gold.filter(col('idade_na_competencia') > 120).count():,}")
print(f"Total de idades invalidas: {idades_invalidas:,}")
print("=" * 55)

## 7. Classificação por faixa etária

A faixa etária facilita comparações e visualizações no BI.

As categorias adotadas são:

- Até 17 anos
- 18 a 24 anos
- 25 a 29 anos
- 30 a 39 anos
- 40 a 49 anos
- 50 a 59 anos
- 60 a 69 anos
- 70 anos ou mais
- Não informado

In [0]:
df_gold = df_gold.withColumn(
    "faixa_etaria",
    when(
        col("idade_na_competencia").isNull(),
        "Não informado"
    )
    .when(
        col("idade_na_competencia") < 0,
        "Idade inválida"
    )
    .when(
        col("idade_na_competencia") <= 17,
        "Até 17 anos"
    )
    .when(
        col("idade_na_competencia") <= 24,
        "18 a 24 anos"
    )
    .when(
        col("idade_na_competencia") <= 29,
        "25 a 29 anos"
    )
    .when(
        col("idade_na_competencia") <= 39,
        "30 a 39 anos"
    )
    .when(
        col("idade_na_competencia") <= 49,
        "40 a 49 anos"
    )
    .when(
        col("idade_na_competencia") <= 59,
        "50 a 59 anos"
    )
    .when(
        col("idade_na_competencia") <= 69,
        "60 a 69 anos"
    )
    .when(
        col("idade_na_competencia") <= 120,
        "70 anos ou mais"
    )
    .otherwise(
        "Idade inválida"
    )
)

display(
    df_gold
    .groupBy("faixa_etaria")
    .count()
    .orderBy("count", ascending=False)
)

## 8. Descrições analíticas do CID

A Silver mantém valores técnicos em `cid_grupo` e `cid_status`.

Na Gold são criadas descrições amigáveis para os consumidores do BI.

A classificação tem foco no objetivo do projeto:

- transtornos mentais;
- doenças osteomusculares;
- outros diagnósticos;
- diagnóstico não informado.

In [0]:
df_gold = (
    df_gold
    .withColumn(
        "cid_grupo_desc",
        when(
            col("cid_grupo") == "mental",
            "Transtornos mentais e comportamentais"
        )
        .when(
            col("cid_grupo") == "osteomuscular",
            "Doenças do sistema osteomuscular"
        )
        .when(
            col("cid_grupo") == "outros",
            "Outras categorias de diagnóstico"
        )
        .when(
            col("cid_grupo") == "nao_informado",
            "Diagnóstico não informado"
        )
        .otherwise(
            "Classificação não mapeada"
        )
    )
    .withColumn(
        "cid_status_desc",
        when(
            col("cid_status") == "informado",
            "CID informado"
        )
        .when(
            col("cid_status") == "nao_informado",
            "CID não informado"
        )
        .otherwise(
            "Situação não mapeada"
        )
    )
)

display(
    df_gold
    .groupBy(
        "cid_grupo",
        "cid_grupo_desc",
        "cid_status",
        "cid_status_desc"
    )
    .count()
    .orderBy("count", ascending=False)
)

## 9. Descrição do tipo de benefício

A coluna técnica `tipo_beneficio`, criada na Silver, recebe um rótulo amigável para uso em segmentações, gráficos e tabelas.

In [0]:
df_gold = df_gold.withColumn(
    "tipo_beneficio_desc",
    when(
        col("tipo_beneficio") == "afastamento",
        "Benefícios relacionados a afastamento"
    )
    .when(
        col("tipo_beneficio") == "aposentadoria",
        "Benefícios de aposentadoria"
    )
    .when(
        col("tipo_beneficio") == "pensao",
        "Benefícios de pensão"
    )
    .when(
        col("tipo_beneficio") == "assistencial",
        "Benefícios assistenciais"
    )
    .when(
        col("tipo_beneficio") == "maternidade",
        "Benefícios relacionados à maternidade"
    )
    .when(
        col("tipo_beneficio") == "outros",
        "Outros tipos de benefício"
    )
    .otherwise(
        "Classificação não mapeada"
    )
)

display(
    df_gold
    .groupBy(
        "tipo_beneficio",
        "tipo_beneficio_desc"
    )
    .count()
    .orderBy("count", ascending=False)
)

## 10. Natureza do afastamento

A natureza identifica se o afastamento é previdenciário ou acidentário.

A classificação principal utiliza:

- espécie 31: previdenciário;
- espécie 91: acidentário;
- demais espécies classificadas como afastamento: outras modalidades;
- benefícios que não são afastamentos: não aplicável.

In [0]:
df_gold = (
    df_gold
    .withColumn(
        "natureza_afastamento",
        when(
            col("especie_cod") == "31",
            "previdenciario"
        )
        .when(
            col("especie_cod") == "91",
            "acidentario"
        )
        .when(
            col("tipo_beneficio") == "afastamento",
            "outras_modalidades"
        )
        .otherwise(
            "nao_aplicavel"
        )
    )
    .withColumn(
        "natureza_afastamento_desc",
        when(
            col("natureza_afastamento") == "previdenciario",
            "Afastamento previdenciário"
        )
        .when(
            col("natureza_afastamento") == "acidentario",
            "Afastamento relacionado ao trabalho"
        )
        .when(
            col("natureza_afastamento") == "outras_modalidades",
            "Outras modalidades relacionadas a afastamento"
        )
        .otherwise(
            "Não se aplica"
        )
    )
)

display(
    df_gold
    .groupBy(
        "especie_cod",
        "especie_desc",
        "tipo_beneficio",
        "natureza_afastamento",
        "natureza_afastamento_desc"
    )
    .count()
    .orderBy("count", ascending=False)
)

## 11. Duração do benefício

A duração é calculada pela diferença entre:

- `dt_dib`: data de início do benefício;
- `dt_dcb`: data de cessação do benefício.

Quando `dt_dcb` não estiver informada, a duração permanecerá nula.

A ausência da data de cessação não significa necessariamente erro. O benefício pode estar ativo ou não possuir cessação definida no arquivo.

In [0]:
df_gold = (
    df_gold
    .withColumn(
        "duracao_beneficio_dias",
        when(
            col("dt_dib").isNotNull()
            & col("dt_dcb").isNotNull(),
            datediff(
                col("dt_dcb"),
                col("dt_dib")
            )
        ).otherwise(None)
    )
    .withColumn(
        "possui_data_cessacao",
        when(
            col("dt_dcb").isNotNull(),
            lit(1)
        ).otherwise(
            lit(0)
        )
    )
    .withColumn(
        "possui_data_cessacao_desc",
        when(
            col("dt_dcb").isNotNull(),
            "Data de cessação informada"
        ).otherwise(
            "Data de cessação não informada"
        )
    )
)

In [0]:
duracoes_negativas = df_gold.filter(
    col("duracao_beneficio_dias") < 0
).count()

duracoes_nulas = df_gold.filter(
    col("duracao_beneficio_dias").isNull()
).count()

print("=" * 60)
print("VALIDACAO DA DURACAO DO BENEFICIO")
print("=" * 60)
print(f"Durações nulas    : {duracoes_nulas:,}")
print(f"Durações negativas: {duracoes_negativas:,}")
print("=" * 60)

## 12. Indicadores auxiliares

São criados campos numéricos binários para facilitar medidas, filtros e validações no BI.

- `qtd_beneficios`: valor 1 para cada registro;
- `ind_afastamento`: 1 quando o benefício foi classificado como afastamento;
- `ind_cid_informado`: 1 quando o CID está informado;
- `ind_saude_mental`: 1 quando o CID pertence ao grupo mental;
- `ind_osteomuscular`: 1 quando o CID pertence ao grupo osteomuscular;
- `ind_acidentario`: 1 quando a natureza é acidentária.

In [0]:
df_gold = (
    df_gold
    .withColumn(
        "qtd_beneficios",
        lit(1)
    )
    .withColumn(
        "ind_afastamento",
        when(
            col("tipo_beneficio") == "afastamento",
            lit(1)
        ).otherwise(lit(0))
    )
    .withColumn(
        "ind_cid_informado",
        when(
            col("cid_status") == "informado",
            lit(1)
        ).otherwise(lit(0))
    )
    .withColumn(
        "ind_saude_mental",
        when(
            col("cid_grupo") == "mental",
            lit(1)
        ).otherwise(lit(0))
    )
    .withColumn(
        "ind_osteomuscular",
        when(
            col("cid_grupo") == "osteomuscular",
            lit(1)
        ).otherwise(lit(0))
    )
    .withColumn(
        "ind_acidentario",
        when(
            col("natureza_afastamento") == "acidentario",
            lit(1)
        ).otherwise(lit(0))
    )
)

In [0]:
df_gold = df_gold.withColumn(
    "_data_processamento_gold",
    current_timestamp()
)

In [0]:
print("=" * 65)
print("VALIDACAO PRE-GRAVACAO DA GOLD PREPARATORIA")
print("=" * 65)

linhas_gold = df_gold.count()
colunas_gold = len(df_gold.columns)

print(f"Linhas Silver : {quantidade_linhas_silver:,}")
print(f"Linhas Gold   : {linhas_gold:,}")
print(f"Colunas Silver: {quantidade_colunas_silver}")
print(f"Colunas Gold  : {colunas_gold}")
print("-" * 65)

if linhas_gold == quantidade_linhas_silver:
    print("RESULTADO DE LINHAS: OK")
else:
    print("RESULTADO DE LINHAS: DIVERGENCIA")

print("=" * 65)

In [0]:
resumo_indicadores = df_gold.agg(
    count(lit(1)).alias("total_registros"),
    spark_sum(col("qtd_beneficios")).alias("qtd_beneficios"),
    spark_sum(col("ind_afastamento")).alias("qtd_afastamentos"),
    spark_sum(col("ind_cid_informado")).alias("qtd_cid_informado"),
    spark_sum(col("ind_saude_mental")).alias("qtd_saude_mental"),
    spark_sum(col("ind_osteomuscular")).alias("qtd_osteomuscular"),
    spark_sum(col("ind_acidentario")).alias("qtd_acidentario")
)

display(resumo_indicadores)

In [0]:
resultado = resumo_indicadores.first()

total_registros = resultado["total_registros"]
qtd_beneficios = resultado["qtd_beneficios"]
qtd_afastamentos = resultado["qtd_afastamentos"]
qtd_cid_informado = resultado["qtd_cid_informado"]
qtd_saude_mental = resultado["qtd_saude_mental"]
qtd_osteomuscular = resultado["qtd_osteomuscular"]
qtd_acidentario = resultado["qtd_acidentario"]

print("=" * 65)
print("VALIDACAO DOS INDICADORES DA GOLD PREPARATORIA")
print("=" * 65)
print(f"{'Total de registros':35} {total_registros:>12,}")
print(f"{'Quantidade de beneficios':35} {qtd_beneficios:>12,}")
print(f"{'Quantidade de afastamentos':35} {qtd_afastamentos:>12,}")
print(f"{'CID informado':35} {qtd_cid_informado:>12,}")
print(f"{'Saude mental':35} {qtd_saude_mental:>12,}")
print(f"{'Osteomuscular':35} {qtd_osteomuscular:>12,}")
print(f"{'Afastamentos acidentarios':35} {qtd_acidentario:>12,}")
print("-" * 65)

erros_validacao = []

if total_registros != qtd_beneficios:
    erros_validacao.append(
        "qtd_beneficios nao corresponde ao total de registros"
    )

if qtd_afastamentos > total_registros:
    erros_validacao.append(
        "qtd_afastamentos e maior que o total de registros"
    )

if qtd_cid_informado > total_registros:
    erros_validacao.append(
        "qtd_cid_informado e maior que o total de registros"
    )

if qtd_saude_mental > qtd_cid_informado:
    erros_validacao.append(
        "qtd_saude_mental e maior que qtd_cid_informado"
    )

if qtd_osteomuscular > qtd_cid_informado:
    erros_validacao.append(
        "qtd_osteomuscular e maior que qtd_cid_informado"
    )

if qtd_acidentario > qtd_afastamentos:
    erros_validacao.append(
        "qtd_acidentario e maior que qtd_afastamentos"
    )

if not erros_validacao:
    print("RESULTADO: OK")
    print("Todas as regras de consistencia foram atendidas.")
else:
    print("RESULTADO: FALHA")
    for erro in erros_validacao:
        print(f"- {erro}")

print("=" * 65)

In [0]:
df_gold = (
    df_gold
    .withColumn(
        "ind_afastamento_saude_mental",
        when(
            (col("tipo_beneficio") == "afastamento")
            & (col("cid_grupo") == "mental"),
            lit(1)
        ).otherwise(lit(0))
    )
    .withColumn(
        "ind_afastamento_osteomuscular",
        when(
            (col("tipo_beneficio") == "afastamento")
            & (col("cid_grupo") == "osteomuscular"),
            lit(1)
        ).otherwise(lit(0))
    )
)

In [0]:
resumo_indicadores = df_gold.agg(
    count(lit(1)).alias("total_registros"),
    spark_sum(col("qtd_beneficios")).alias("qtd_beneficios"),
    spark_sum(col("ind_afastamento")).alias("qtd_afastamentos"),
    spark_sum(col("ind_cid_informado")).alias("qtd_cid_informado"),
    spark_sum(col("ind_saude_mental")).alias("qtd_saude_mental"),
    spark_sum(col("ind_osteomuscular")).alias("qtd_osteomuscular"),
    spark_sum(col("ind_acidentario")).alias("qtd_acidentario"),
    spark_sum(
        col("ind_afastamento_saude_mental")
    ).alias("qtd_afastamentos_saude_mental"),
    spark_sum(
        col("ind_afastamento_osteomuscular")
    ).alias("qtd_afastamentos_osteomusculares")
)

display(resumo_indicadores)

In [0]:
resumo_afastamentos = df_gold.agg(
    spark_sum(
        col("ind_afastamento")
    ).alias("qtd_afastamentos"),

    spark_sum(
        col("ind_afastamento_saude_mental")
    ).alias("qtd_afastamentos_saude_mental"),

    spark_sum(
        col("ind_afastamento_osteomuscular")
    ).alias("qtd_afastamentos_osteomusculares"),

    spark_sum(
        col("ind_acidentario")
    ).alias("qtd_afastamentos_acidentarios")
)

display(resumo_afastamentos)


In [0]:
VALIDACOES_SEMANTICAS = {
    "cid_grupo_desc": "Classificação não mapeada",
    "cid_status_desc": "Situação não mapeada",
    "tipo_beneficio_desc": "Classificação não mapeada"
}

print("=" * 70)
print("VALIDACAO DE CLASSIFICACOES SEMANTICAS")
print("=" * 70)

total_nao_mapeado = 0

for coluna, valor_nao_mapeado in VALIDACOES_SEMANTICAS.items():
    quantidade = (
        df_gold
        .filter(col(coluna) == valor_nao_mapeado)
        .count()
    )

    total_nao_mapeado += quantidade
    status = "OK" if quantidade == 0 else "ATENCAO"

    print(
        f"{coluna:<35} "
        f"{quantidade:>12,} "
        f"{status:>12}"
    )

print("-" * 70)

if total_nao_mapeado == 0:
    print("RESULTADO: Todas as classificacoes foram mapeadas.")
else:
    print(
        "RESULTADO: Existem classificacoes que precisam "
        "de revisao."
    )

print("=" * 70)

## 13. Gravação da Gold preparatória

A tabela é persistida em formato Delta no schema Gold.

O modo `overwrite` permite reconstruir integralmente a camada a partir da Silver, preservando a reprodutibilidade do pipeline.

A tabela mantém o mesmo grão da Silver e acrescenta atributos semânticos e indicadores auxiliares para BI.

In [0]:
(
    df_gold
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DESTINO)
)

print(f"Tabela gravada: {TABELA_DESTINO}")

In [0]:
df_gold_persistida = spark.table(TABELA_DESTINO)

linhas_silver = df_silver.count()
linhas_gold = df_gold_persistida.count()

colunas_silver = len(df_silver.columns)
colunas_gold = len(df_gold_persistida.columns)

soma_qtd_beneficios = (
    df_gold_persistida
    .agg(
        spark_sum(
            col("qtd_beneficios")
        ).alias("total")
    )
    .first()["total"]
)

print("=" * 70)
print("VALIDACAO SILVER VS GOLD PREPARATORIA")
print("=" * 70)
print(
    f"{'Metrica':<32}"
    f"{'Silver':>15}"
    f"{'Gold':>15}"
)
print("-" * 70)
print(
    f"{'Linhas':<32}"
    f"{linhas_silver:>15,}"
    f"{linhas_gold:>15,}"
)
print(
    f"{'Colunas':<32}"
    f"{colunas_silver:>15}"
    f"{colunas_gold:>15}"
)
print(
    f"{'Soma qtd_beneficios':<32}"
    f"{'Nao aplicavel':>15}"
    f"{soma_qtd_beneficios:>15,}"
)
print("-" * 70)

erros = []

if linhas_silver != linhas_gold:
    erros.append(
        "Quantidade de linhas da Gold difere da Silver."
    )

if soma_qtd_beneficios != linhas_gold:
    erros.append(
        "Soma de qtd_beneficios difere do total da Gold."
    )

if colunas_gold <= colunas_silver:
    erros.append(
        "A Gold nao apresentou as colunas semanticas esperadas."
    )

if not erros:
    print("RESULTADO: OK")
    print(
        "A Gold preservou o grao e os indicadores "
        "foram persistidos corretamente."
    )
else:
    print("RESULTADO: FALHA")
    for erro in erros:
        print(f"- {erro}")

print("=" * 70)

In [0]:
%sql
COMMENT ON TABLE afastamento_inss.gold.prep_beneficios_bi
IS 'Camada semantica detalhada de beneficios concedidos pelo INSS, preparada para consumo em BI. Mantem uma linha por registro de beneficio e inclui classificacoes, descricoes, atributos temporais e indicadores auxiliares.';

In [0]:
%sql
ALTER TABLE afastamento_inss.gold.prep_beneficios_bi
ALTER COLUMN qtd_beneficios
COMMENT 'Medida unitaria com valor 1 para contagem aditiva de registros de beneficios.';

ALTER TABLE afastamento_inss.gold.prep_beneficios_bi
ALTER COLUMN tipo_beneficio
COMMENT 'Classificacao tecnica do beneficio em afastamento, aposentadoria, pensao, assistencial, maternidade ou outros.';

ALTER TABLE afastamento_inss.gold.prep_beneficios_bi
ALTER COLUMN tipo_beneficio_desc
COMMENT 'Descricao amigavel da classificacao do tipo de beneficio para consumo em BI.';

ALTER TABLE afastamento_inss.gold.prep_beneficios_bi
ALTER COLUMN cid_grupo
COMMENT 'Grupo analitico do CID em mental, osteomuscular, outros ou nao informado.';

ALTER TABLE afastamento_inss.gold.prep_beneficios_bi
ALTER COLUMN cid_grupo_desc
COMMENT 'Descricao amigavel do grupo analitico do CID.';

ALTER TABLE afastamento_inss.gold.prep_beneficios_bi
ALTER COLUMN natureza_afastamento
COMMENT 'Natureza do afastamento em previdenciario, acidentario, outras modalidades ou nao aplicavel.';

ALTER TABLE afastamento_inss.gold.prep_beneficios_bi
ALTER COLUMN duracao_beneficio_dias
COMMENT 'Quantidade de dias entre a data de inicio e a data de cessacao do beneficio, quando ambas estiverem informadas.';

ALTER TABLE afastamento_inss.gold.prep_beneficios_bi
ALTER COLUMN idade_na_competencia
COMMENT 'Idade aproximada do beneficiario no primeiro dia do mes da competencia.';

ALTER TABLE afastamento_inss.gold.prep_beneficios_bi
ALTER COLUMN faixa_etaria
COMMENT 'Faixa etaria derivada da idade aproximada na competencia.';

ALTER TABLE afastamento_inss.gold.prep_beneficios_bi
ALTER COLUMN ind_afastamento
COMMENT 'Indicador binario: 1 para beneficio classificado como afastamento e 0 para os demais.';

ALTER TABLE afastamento_inss.gold.prep_beneficios_bi
ALTER COLUMN ind_acidentario
COMMENT 'Indicador binario: 1 para afastamento de natureza acidentaria e 0 para os demais.';

### Indicadores específicos dos registros classificados como afastamento

| Indicador | Quantidade | Percentual sobre os registros de afastamento |
|---|---:|---:|
| Total de registros classificados como afastamento | 185.412 | 100,00% |
| Afastamentos associados a transtornos mentais e comportamentais | 19.572 | 10,56% |
| Afastamentos associados a doenças do sistema osteomuscular | 36.251 | 19,55% |
| Afastamentos de natureza acidentária | 13.428 | 7,24% |

Os percentuais foram calculados utilizando como denominador os **185.412 registros de benefícios classificados como afastamento**.

Os resultados representam a distribuição interna dos registros presentes no arquivo analisado. A base contém benefícios concedidos pelo INSS e não permite afirmar que cada registro corresponde a uma pessoa beneficiária única, pois não foi identificado um código individual de benefício ou beneficiário que permita confirmar essa unicidade.

Consequentemente, os indicadores não representam:

- incidência de afastamentos na população;
- percentual de trabalhadores afastados;
- quantidade de pessoas únicas;
- taxa de afastamento por setor econômico;
- prevalência das condições de saúde na população.

Para calcular taxas de afastamento seria necessário utilizar um denominador externo, como a quantidade de vínculos empregatícios da RAIS ou do Novo CAGED no mesmo período e recorte geográfico ou econômico.

A presença ou ausência de CNAE também não confirma a condição laboral da pessoa beneficiária. O CNAE representa a atividade econômica associada ao empregador ou estabelecimento quando essa informação está disponível e é aplicável ao benefício. Por isso, análises por CNAE serão restritas aos registros com código informado e deverão apresentar explicitamente a cobertura dessa informação.

A classificação de natureza acidentária representa a categoria administrativa identificada a partir da espécie do benefício. Essa classificação indica reconhecimento previdenciário de relação com acidente ou doença do trabalho, mas não deve ser generalizada para todos os registros que possuem diagnóstico de saúde mental ou osteomuscular.